In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import scienceplots
import scipy
from dotenv import load_dotenv
import os

import tensorstore as ts

load_dotenv()
PATH = os.getenv("ROOT_PATH")

plt.style.use(['science'])

def format_ax(ax):
  for spine in ax.spines.values():
    spine.set_linewidth(1.2)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.spines['bottom'].set_visible(False)
  ax.tick_params(which='minor', length=0)
  ax.tick_params(axis='both', labelsize=12)
  for spine in ax.spines.values():
    spine.set_visible(False)
  ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
  ax.tick_params(axis='y', which='both', left=False, right=False, direction="out", width=1.2)

In [ ]:
subject_ids = ["01", "02", "03", "04", "06", "12", "13", "14", "15", "16"]
n_neurons = 50_000
x_pretrain = []

for subject_id in subject_ids:
  ds_janelia = ts.open({
      'open': True,
      'driver': 'zarr3',
      'kvstore': f'file://{PATH}/ts_files/subject_{subject_id}_traces.zarr'
  }).result()

  x = ds_janelia.read().result()
  print(x.shape)
  ix = np.random.choice(np.arange(x.shape[-1]), size=n_neurons, replace=False)
  x_pretrain.append(x[:, ix])

In [ ]:
x_pretrain = np.concatenate(x_pretrain, axis=0)

In [ ]:
x_pretrain.shape

In [ ]:
PATH_STORE = "/mnt/storage/misc/zapbench/data/ts_files"
spec = {
    'driver': 'zarr3',
    'kvstore': {'driver': 'file', 'path': f'{PATH_STORE}/janelia_pretrain_traces.zarr'},
    'metadata': {
        'shape': list(x_pretrain.shape),
        'chunk_grid': {'name': 'regular', 'configuration': {'chunk_shape': [512, min(512, x_pretrain.shape[-1])]}},
        'chunk_key_encoding': {'name': 'default'},
        'codecs': [{'name': 'bytes', 'configuration': {'endian': 'little'}}],
        'data_type': 'float32',
        'fill_value': 0.0
    }
}
ds = ts.open(spec, create=True).result()
ds[...] = x_pretrain

Count neurons and t

In [ ]:
subject_ids = ["01", "02", "03", "04", "05", "06", "07", "12", "13", "14", "15", "16", "17"]
neurons = []
t = 0
for subject_id in subject_ids:
  ds_janelia = ts.open({
      'open': True,
      'driver': 'zarr3',
      'kvstore': f'file://{PATH}/ts_files/subject_{subject_id}_traces.zarr'
  }).result()

  x = ds_janelia.read().result()
  t += x.shape[0]
  neurons.append(x.shape[-1])
neurons = np.array(neurons)

In [ ]:
t, neurons.min(), neurons.max()

In [ ]:
ds_janelia = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

In [ ]:
ds_janelia.shape

In [ ]:
subject_ids = ["01", "02", "03", "04", "06", "12", "13", "14", "15", "16"]
stimuli_pretrain = []

for subject_id in subject_ids:
  ds_janelia = ts.open({
      'open': True,
      'driver': 'zarr3',
      'kvstore': f'file://{PATH}/ts_files/subject_{subject_id}_stimuli.zarr'
  }).result()

  x = ds_janelia.read().result()
  print(x.shape)
  stimuli_pretrain.append(x)

In [ ]:
stimuli_pretrain = np.concatenate(stimuli_pretrain, axis=0)

In [ ]:
PATH_STORE = "/mnt/storage/misc/zapbench/data/ts_files"
spec = {
    'driver': 'zarr3',
    'kvstore': {'driver': 'file', 'path': f'{PATH_STORE}/janelia_pretrain_stimuli.zarr'},
    'metadata': {
        'shape': list(stimuli_pretrain.shape),
        'chunk_grid': {'name': 'regular', 'configuration': {'chunk_shape': [512, min(512, stimuli_pretrain.shape[-1])]}},
        'chunk_key_encoding': {'name': 'default'},
        'codecs': [{'name': 'bytes', 'configuration': {'endian': 'little'}}],
        'data_type': 'float32',
        'fill_value': 0.0
    }
}
ds = ts.open(spec, create=True).result()
ds[...] = stimuli_pretrain